# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** Refresh / Content Opportunity Scoring

**Method:** Gradient Boosting (scikit-learn `HistGradientBoostingClassifier`).

I chose Gradient Boosting because the refresh-priority problem combines several numeric SEO and engagement signals, and their relationship with a future decline is unlikely to be perfectly linear. A boosted-tree model can learn interactions between CTR, Google Search position, organic sessions, and GA4 engagement without requiring a hand-written threshold for every combination.

The model is used for **ranking and decision support**, not as proof that a content refresh will improve performance.

For an honest target, this notebook uses a **near-term organic-session decline proxy**: a page is positive when its next observed organic-session value falls by at least 20% within the next 7 days. This is a measurable future outcome available from the warehouse data, but it is not a direct "refresh succeeded" label.

In [2]:
# Imports and memory-conscious data loading
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from google.colab import files

SEED = 42

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("Upload the fact_content_daily_performance_sample.parquet file.")

filename = list(uploaded.keys())[0]

# Read only the fields needed for this assignment.
required_cols = [
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_engaged_sessions", "sessions_organic"
]

schema_cols = pq.ParquetFile(filename).schema.names
missing = [c for c in required_cols if c not in schema_cols]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = pq.read_table(filename, columns=required_cols).to_pandas()

df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")

numeric_cols = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_engaged_sessions", "sessions_organic"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# CTR is calculated from the raw GSC counts.
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"] * 100,
    0.0
)

print("Rows:", len(df))
print("Columns loaded:", len(df.columns))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())
display(df.head())

# Grain check from ML-04: one content/client/date observation.
grain_cols = ["client_hash_id", "content_hash_id", "report_date"]
duplicate_rows = df.duplicated(grain_cols).sum()
print("Duplicate client-content-date rows:", int(duplicate_rows))

if duplicate_rows:
    df = df.drop_duplicates(grain_cols, keep="last").copy()
    print("Rows after duplicate removal:", len(df))


Saving fact_content_daily_performance_sample.parquet to fact_content_daily_performance_sample.parquet
Rows: 11694072
Columns loaded: 9
Date range: 2026-06-01 00:00:00 to 2026-06-30 00:00:00


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,sessions_organic,ctr
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,0,0,NaN,0.0,0.0,0.0
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,0,0,NaN,0.0,0.0,0.0
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,0,0,NaN,0.0,0.0,0.0
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,0,0,NaN,0.0,0.0,0.0
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,0,0,NaN,0.0,0.0,0.0


Duplicate client-content-date rows: 6390
Rows after duplicate removal: 11687682


## 2. Split design

I use a **time-aware split**. Features come from the current observation and the target is a future organic-session decline, so training on earlier observations and validating on later observations is more realistic than randomly mixing dates.

The split date is calculated from the labeled observations. The baseline thresholds are also calculated from the training portion only and then applied unchanged to validation. This avoids using validation-period distributions to tune the comparison.

The notebook also checks the client/content/date grain and removes duplicate rows only if duplicates are actually present.

In [3]:
# Create a future, leakage-safe proxy target.
# Positive = next observed organic sessions are at least 20% lower,
# with the next observation no more than 7 days after the current one.

df = df.sort_values(["client_hash_id", "content_hash_id", "report_date"]).reset_index(drop=True)

group_cols = ["client_hash_id", "content_hash_id"]
df["next_report_date"] = df.groupby(group_cols)["report_date"].shift(-1)
df["next_sessions_organic"] = df.groupby(group_cols)["sessions_organic"].shift(-1)

df["days_to_next"] = (df["next_report_date"] - df["report_date"]).dt.days

eligible = (
    df["days_to_next"].between(1, 7, inclusive="both")
    & df["sessions_organic"].notna()
    & df["next_sessions_organic"].notna()
    & (df["sessions_organic"] > 0)
)

df["future_decline"] = np.nan
df.loc[eligible, "future_decline"] = (
    df.loc[eligible, "next_sessions_organic"]
    <= 0.80 * df.loc[eligible, "sessions_organic"]
).astype(int)

model_df = df.loc[eligible].copy()
model_df["future_decline"] = model_df["future_decline"].astype(int)

print("Eligible labeled observations:", len(model_df))
print("Positive future-decline observations:", int(model_df["future_decline"].sum()))
print("Positive rate:", round(model_df["future_decline"].mean(), 4))
print("Maximum target gap (days):", int(model_df["days_to_next"].max()))

if len(model_df) < 200:
    raise ValueError("Too few eligible observations for a meaningful train/validation comparison.")

display(
    model_df[[
        "report_date", "gsc_impressions", "gsc_clicks", "ctr",
        "gsc_avg_position", "ga4_engaged_sessions",
        "sessions_organic", "next_sessions_organic", "future_decline"
    ]].head()
)


Eligible labeled observations: 331697
Positive future-decline observations: 231613
Positive rate: 0.6983
Maximum target gap (days): 1


,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,next_sessions_organic,future_decline
94,2026-06-20,0,0,0.0,NaN,0.0,2.0,0.0,1
816,2026-06-13,0,0,0.0,NaN,0.0,1.0,0.0,1
1029,2026-06-24,0,0,0.0,NaN,1.0,2.0,0.0,1
1138,2026-06-03,0,0,0.0,NaN,0.0,1.0,0.0,1
1160,2026-06-25,0,0,0.0,NaN,0.0,1.0,0.0,1


## 3. Train + compare vs my baseline

The model and the Week-4 rule are evaluated on the **same validation rows** and against the **same future-decline proxy**.

The primary metric is **Precision@K**, because the business action is to review a limited ranked queue of pages first. I report Precision@10, Precision@20 and Precision@50 where enough validation rows are available.

The Week-4 baseline uses the same four signals from ML-07:

- low CTR
- poor Google Search average position
- low GA4 engaged sessions
- low organic sessions

The thresholds are learned from the training split and then frozen for validation.

In [4]:
# Time-aware split
split_date = model_df["report_date"].quantile(0.80)

train = model_df[model_df["report_date"] < split_date].copy()
valid = model_df[model_df["report_date"] >= split_date].copy()

print("Split date:", split_date)
print("Train rows:", len(train))
print("Validation rows:", len(valid))
print("Train dates:", train["report_date"].min(), "to", train["report_date"].max())
print("Validation dates:", valid["report_date"].min(), "to", valid["report_date"].max())
print("Train positive rate:", round(train["future_decline"].mean(), 4))
print("Validation positive rate:", round(valid["future_decline"].mean(), 4))

if train["future_decline"].nunique() < 2 or valid["future_decline"].nunique() < 2:
    raise ValueError("The time split does not contain both target classes. Inspect the date distribution before submission.")

# Features available at prediction time.
feature_cols = [
    "gsc_impressions", "gsc_clicks", "ctr",
    "gsc_avg_position", "ga4_engaged_sessions", "sessions_organic"
]

X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train["future_decline"]

X_valid = valid[feature_cols].replace([np.inf, -np.inf], np.nan)
y_valid = valid["future_decline"]

print("\nFeature columns:")
print(feature_cols)

# Train Gradient Boosting.
# A bounded training sample keeps Colab memory use predictable on the 11M+ row warehouse sample.
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

MAX_TRAIN_ROWS = 300_000

if len(X_train) > MAX_TRAIN_ROWS:
    rng = np.random.RandomState(SEED)
    keep_idx = rng.choice(len(X_train), size=MAX_TRAIN_ROWS, replace=False)
    X_fit = X_train.iloc[keep_idx]
    y_fit = y_train.iloc[keep_idx]
else:
    X_fit = X_train
    y_fit = y_train

model = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=250,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=SEED
    )
)

model.fit(X_fit, y_fit)

model_prob = model.predict_proba(X_valid)[:, 1]

print("Training rows used:", len(X_fit))
print("Validation rows:", len(X_valid))
print("Validation ROC-AUC:", round(roc_auc_score(y_valid, model_prob), 4))
print("Validation Average Precision:", round(average_precision_score(y_valid, model_prob), 4))

# Reproduce the Week-4 baseline without validation leakage.
# Thresholds are learned from the training split only.

ctr_q = train["ctr"].quantile(0.25)
pos_q = train["gsc_avg_position"].quantile(0.75)
eng_q = train["ga4_engaged_sessions"].quantile(0.25)
org_q = train["sessions_organic"].quantile(0.25)

def baseline_score(frame):
    score = pd.Series(0, index=frame.index, dtype="int16")
    score += (frame["ctr"] < ctr_q).astype("int16") * 40
    score += (frame["gsc_avg_position"] > pos_q).astype("int16") * 30
    score += (frame["ga4_engaged_sessions"] < eng_q).astype("int16") * 20
    score += (frame["sessions_organic"] < org_q).astype("int16") * 10
    return score

valid["baseline_score"] = baseline_score(valid)

def precision_at_k(y_true, ranking_score, k):
    k = min(k, len(y_true))
    order = np.argsort(np.asarray(ranking_score))[::-1][:k]
    return float(np.asarray(y_true)[order].mean())

ks = [10, 20, 50]
rows = []

for k in ks:
    if len(valid) >= k:
        rows.append({
            "Method": "Week-4 baseline",
            "K": k,
            "Precision@K": precision_at_k(y_valid, valid["baseline_score"], k)
        })
        rows.append({
            "Method": "Gradient Boosting",
            "K": k,
            "Precision@K": precision_at_k(y_valid, model_prob, k)
        })

comparison = pd.DataFrame(rows)
display(comparison)

print("\nTraining-only thresholds used by the baseline:")
print("CTR 25th percentile:", round(float(ctr_q), 4))
print("Position 75th percentile:", round(float(pos_q), 4))
print("Engaged sessions 25th percentile:", round(float(eng_q), 4))
print("Organic sessions 25th percentile:", round(float(org_q), 4))


Split date: 2026-06-24 00:00:00
Train rows: 261237
Validation rows: 70460
Train dates: 2026-06-01 00:00:00 to 2026-06-23 00:00:00
Validation dates: 2026-06-24 00:00:00 to 2026-06-29 00:00:00
Train positive rate: 0.712
Validation positive rate: 0.6475

Feature columns:
['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic']
Training rows used: 261237
Validation rows: 70460
Validation ROC-AUC: 0.4805
Validation Average Precision: 0.6567


,Method,K,Precision@K
0,Week-4 baseline,10,0.30
1,Gradient Boosting,10,1.00
2,Week-4 baseline,20,0.30
3,Gradient Boosting,20,1.00
4,Week-4 baseline,50,0.32
5,Gradient Boosting,50,1.00



Training-only thresholds used by the baseline:
CTR 25th percentile: 0.1603
Position 75th percentile: 13.0349
Engaged sessions 25th percentile: 0.0
Organic sessions 25th percentile: 1.0


## 4. Errors and interpretation

A ranking model can be useful even when it is not perfect. I inspect false positives and false negatives on the validation set and compare the model's top-ranked pages with the baseline.

Feature importance is used as a directional interpretation tool. It tells us which input variables the fitted model relied on most in this sample; it does **not** establish causation.

The main limitations are important: the target is a proxy rather than a direct refresh-success label, the next observation may not represent a perfectly fixed calendar interval, and external SEO factors are not represented in this warehouse sample.

In [5]:
# Build a review table showing model ranking, baseline ranking and the future outcome.
valid["model_score"] = model_prob
valid["model_rank"] = valid["model_score"].rank(method="first", ascending=False).astype(int)
valid["baseline_rank"] = valid["baseline_score"].rank(method="first", ascending=False).astype(int)

review_cols = [
    "report_date", "content_hash_id", "client_hash_id",
    "model_score", "baseline_score", "future_decline",
    "ctr", "gsc_avg_position", "ga4_engaged_sessions", "sessions_organic",
    "next_sessions_organic"
]

top_model = valid.sort_values("model_score", ascending=False).head(20)[review_cols].copy()
top_baseline = valid.sort_values("baseline_score", ascending=False).head(20)[review_cols].copy()

print("Top model-ranked validation observations:")
display(top_model)

print("Top baseline-ranked validation observations:")
display(top_baseline)

# Disagreement/error categories for the model at K=20.
k = min(20, len(valid))
top20_idx = valid.sort_values("model_score", ascending=False).head(k).index
pred_top20 = pd.Series(0, index=valid.index)
pred_top20.loc[top20_idx] = 1

valid["model_top20"] = pred_top20.astype(int)

valid["error_type"] = np.select(
    [
        (valid["model_top20"] == 1) & (valid["future_decline"] == 1),
        (valid["model_top20"] == 1) & (valid["future_decline"] == 0),
        (valid["model_top20"] == 0) & (valid["future_decline"] == 1)
    ],
    ["True positive", "False positive", "False negative"],
    default="True negative"
)

print("Model top-20 error summary:")
display(valid["error_type"].value_counts().to_frame("count"))

print("\nFalse positives — high model score but no observed decline:")
display(
    valid[valid["error_type"] == "False positive"]
    .sort_values("model_score", ascending=False)
    .head(10)[review_cols]
)

print("\nFalse negatives — observed decline but not in model top-20:")
display(
    valid[valid["error_type"] == "False negative"]
    .sort_values("model_score", ascending=False)
    .head(10)[review_cols]
)

# Directional feature interpretation using permutation importance.
from sklearn.inspection import permutation_importance

sample_n = min(10_000, len(X_valid))
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(X_valid), size=sample_n, replace=False)

perm = permutation_importance(
    model,
    X_valid.iloc[sample_idx],
    y_valid.iloc[sample_idx],
    scoring="average_precision",
    n_repeats=3,
    random_state=SEED,
    n_jobs=-1
)

importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "mean_importance": perm.importances_mean,
        "std_importance": perm.importances_std
    })
    .sort_values("mean_importance", ascending=False)
    .reset_index(drop=True)
)

display(importance)

print(
    "Interpretation: larger permutation importance means that shuffling that feature "
    "reduced validation Average Precision more in this sample. This is directional, "
    "not causal evidence."
)

# Compact conclusion generated from the actual comparison.
for k in [10, 20, 50]:
    if len(valid) >= k:
        p_base = precision_at_k(y_valid, valid["baseline_score"], k)
        p_model = precision_at_k(y_valid, model_prob, k)
        delta = p_model - p_base
        print(
            f"Precision@{k}: model={p_model:.4f}, baseline={p_base:.4f}, "
            f"difference={delta:+.4f}"
        )

print("\nDecision-support conclusion:")
print(
    "The Gradient Boosting model should be preferred only where its validation "
    "Precision@K is better and the difference is meaningful enough to justify "
    "the added complexity. Otherwise, the transparent Week-4 rule remains a "
    "reasonable baseline."
)


Top model-ranked validation observations:


,report_date,content_hash_id,client_hash_id,model_score,baseline_score,future_decline,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,next_sessions_organic
8996641,2026-06-26,content_6740cf456a4101ac,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996757,2026-06-26,content_6762801d4726f0db,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8928868,2026-06-26,content_000257fd4bd9d9fb,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996728,2026-06-26,content_6754fa328e5e2730,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8928926,2026-06-26,content_001d68f5ddb53c69,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996670,2026-06-26,content_674be99eeacaa73b,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8928984,2026-06-26,content_003023a846c0ecdd,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996322,2026-06-26,content_66d9000627e254a0,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996351,2026-06-26,content_66d957175a497a8f,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8996409,2026-06-26,content_66e4c715982ccbcc,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0


Top baseline-ranked validation observations:


,report_date,content_hash_id,client_hash_id,model_score,baseline_score,future_decline,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,next_sessions_organic
9016216,2026-06-26,content_87cc77bd794bdf28,client_a60a11451483af1c,0.999167,70,1,0.0,81.000000,0.0,7.0,0.0
2400382,2026-06-26,content_b82f5f6d8a977171,client_23a62021009f63c4,0.940933,70,1,0.0,40.750000,0.0,2.0,0.0
2400625,2026-06-29,content_b85873e80b3bb4f6,client_23a62021009f63c4,0.947927,70,1,0.0,22.927711,0.0,3.0,0.0
9039299,2026-06-25,content_ac64feb7b14cf79d,client_a60a11451483af1c,0.998996,70,0,0.0,70.000000,0.0,7.0,7.0
2400621,2026-06-25,content_b85873e80b3bb4f6,client_23a62021009f63c4,0.870083,70,1,0.0,21.935065,0.0,2.0,1.0
2400620,2026-06-24,content_b85873e80b3bb4f6,client_23a62021009f63c4,0.864682,70,0,0.0,25.781609,0.0,2.0,2.0
2400563,2026-06-27,content_b851364479eb57c9,client_23a62021009f63c4,0.914122,70,1,0.0,45.178571,0.0,1.0,0.0
2400502,2026-06-26,content_b8465fead1d8e67d,client_23a62021009f63c4,0.863419,70,1,0.0,18.607143,0.0,2.0,0.0
2400415,2026-06-29,content_b82fe9f8a98d53f6,client_23a62021009f63c4,0.888155,70,1,0.0,39.120370,0.0,2.0,0.0
2400412,2026-06-26,content_b82fe9f8a98d53f6,client_23a62021009f63c4,0.903640,70,1,0.0,41.617647,0.0,2.0,0.0


Model top-20 error summary:


...

,report_date,content_hash_id,client_hash_id,model_score,baseline_score,future_decline,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,next_sessions_organic



False negatives — observed decline but not in model top-20:


,report_date,content_hash_id,client_hash_id,model_score,baseline_score,future_decline,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,next_sessions_organic
8928955,2026-06-26,content_001f61f497cc8a93,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8928897,2026-06-26,content_000a59f75d37d1ef,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929042,2026-06-26,content_005a71ef9c9431c1,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929071,2026-06-26,content_005fdc47efde61e6,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929100,2026-06-26,content_00636bb58fb29cfa,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929129,2026-06-26,content_006e9559b6962403,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929158,2026-06-26,content_0089f45a08168d98,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929187,2026-06-26,content_00aae18b02f6dcc6,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929216,2026-06-26,content_00ad7f2a7ea677e4,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0
8929274,2026-06-26,content_00b7ccb3fec3922f,client_a60a11451483af1c,0.999167,70,1,0.0,81.0,0.0,7.0,0.0


,feature,mean_importance,std_importance
0,gsc_impressions,0.099095,0.000832
1,sessions_organic,0.041070,0.002252
2,gsc_clicks,0.028264,0.000595
3,gsc_avg_position,0.020433,0.001688
4,ctr,0.014599,0.000144
5,ga4_engaged_sessions,0.001506,0.000225


Interpretation: larger permutation importance means that shuffling that feature reduced validation Average Precision more in this sample. This is directional, not causal evidence.
Precision@10: model=1.0000, baseline=0.3000, difference=+0.7000
Precision@20: model=1.0000, baseline=0.3000, difference=+0.7000
Precision@50: model=1.0000, baseline=0.3200, difference=+0.6800

Decision-support conclusion:
The Gradient Boosting model should be preferred only where its validation Precision@K is better and the difference is meaningful enough to justify the added complexity. Otherwise, the transparent Week-4 rule remains a reasonable baseline.


## Self-check

- [x] Method choice is explained and fits the Refresh / Content Opportunity Scoring lane.
- [x] Split is time-aware and the baseline thresholds are learned only from training data.
- [x] Model and Week-4 baseline use the same validation rows and the same Precision@K metric.
- [x] Error analysis and feature interpretation are included.
- [x] No client names, URLs, or private queries are used; only anonymized IDs are displayed.
- [ ] After opening the notebook in Colab, run **Runtime → Run all** with the Parquet dataset uploaded.
- [ ] Commit the executed notebook under `work/notebooks/w05_model.ipynb`.
